## Configurando o Banco em Memória e Ingerindo os Dados

In [1]:
!pip install sqlalchemy

  Using cached sqlalchemy-2.0.50-cp312-cp312-win_amd64.whl.metadata (9.8 kB)
  Using cached greenlet-3.5.1-cp312-cp312-win_amd64.whl.metadata (3.9 kB)
Using cached sqlalchemy-2.0.50-cp312-cp312-win_amd64.whl (2.1 MB)
Using cached greenlet-3.5.1-cp312-cp312-win_amd64.whl (238 kB)



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sqlalchemy import create_engine
import pandas as pd

# 1. Criando um banco de dados SQLite em memória (não ocupa espaço em disco e limpa ao fechar)
engine = create_engine('sqlite:///:memory:')

# 2. Lendo o dataset limpo que geramos no passo anterior
df_limpo = pd.read_csv('../data/dataset_credito_limpo.csv')

# 3. Ingerindo os dados do DataFrame para o banco de dados simulado
# Isso cria a tabela 'clientes_credito' automaticamente com a estrutura correta
df_limpo.to_sql('clientes_credito', con=engine, index=False, if_exists='replace')

print("Banco de dados relacional simulado com sucesso!")
print("Tabela 'clientes_credito' pronta para receber consultas SQL.")

✅ Banco de dados relacional simulado com sucesso!
Tabela 'clientes_credito' pronta para receber consultas SQL.


## Faixa Etária e Receita

In [3]:
# Query para agrupar clientes por faixas de idade e calcular métricas financeiras
query_faixa_etaria = """
SELECT 
    CASE 
        WHEN idade < 30 THEN '1. Até 29 anos'
        WHEN idade BETWEEN 30 AND 39 THEN '2. 30 a 39 anos'
        WHEN idade BETWEEN 40 AND 49 THEN '3. 40 a 49 anos'
        WHEN idade BETWEEN 50 AND 59 THEN '4. 50 a 59 anos'
        ELSE '5. 60+ anos'
    END AS faixa_etaria,
    COUNT(id_cliente) AS total_clientes,
    ROUND(AVG(valor_total_transacoes), 2) AS ticket_medio_gasto,
    ROUND(SUM(valor_total_transacoes), 2) AS receita_total_gerada
FROM 
    clientes_credito
GROUP BY 
    faixa_etaria
ORDER BY 
    receita_total_gerada DESC;
"""

# Executando a query e exibindo o resultado
df_faixa_etaria = pd.read_sql_query(query_faixa_etaria, engine)
display(df_faixa_etaria)

,faixa_etaria,total_clientes,ticket_medio_gasto,receita_total_gerada
0,3. 40 a 49 anos,4561,4633.81,21134826.0
1,4. 50 a 59 anos,2998,4298.21,12886044.0
2,2. 30 a 39 anos,1841,4275.85,7871836.0
3,5. 60+ anos,532,3527.43,1876592.0
4,1. Até 29 anos,195,4260.94,830884.0


## Performance dos Produtos (Cartões)

In [4]:
query_produtos = """
SELECT 
    categoria_cartao,
    COUNT(id_cliente) AS volume_clientes,
    ROUND(SUM(limite_credito), 2) AS limite_total_concedido,
    ROUND(SUM(valor_total_transacoes), 2) AS volume_transacionado,
    ROUND((SUM(valor_total_transacoes) / SUM(limite_credito)) * 100, 2) AS pct_utilizacao_limite
FROM 
    clientes_credito
GROUP BY 
    categoria_cartao
ORDER BY 
    volume_transacionado DESC;
"""

df_produtos = pd.read_sql_query(query_produtos, engine)
display(df_produtos)

,categoria_cartao,volume_clientes,limite_total_concedido,volume_transacionado,pct_utilizacao_limite
0,Blue,9436,69484628.1,39870938.0,57.38
1,Silver,555,14029199.0,3657718.0,26.07
2,Gold,116,3296299.0,891531.0,27.05
3,Platinum,20,605669.0,179995.0,29.72


## Alvos de Churn com Alta Rentabilidade

In [6]:
query_risco_churn = """
SELECT 
    id_cliente,
    idade,
    categoria_cartao,
    valor_total_transacoes,
    qtd_total_transacoes,
    status_cliente
FROM 
    clientes_credito
WHERE 
    valor_total_transacoes > 4000 -- Calibragem: Reduzimos o corte financeiro
    AND qtd_total_transacoes < 55 -- Calibragem: Aumentamos a margem de inatividade
    AND status_cliente = 'Existing Customer'
ORDER BY 
    valor_total_transacoes DESC
LIMIT 10;
"""

df_risco_churn = pd.read_sql_query(query_risco_churn, engine)
display(df_risco_churn)

,id_cliente,idade,categoria_cartao,valor_total_transacoes,qtd_total_transacoes,status_cliente
0,789345408,39,Blue,4983,53,Existing Customer
1,708699108,39,Blue,4890,53,Existing Customer
2,719038008,48,Blue,4862,41,Existing Customer
3,718921683,49,Blue,4816,49,Existing Customer
4,712311258,48,Blue,4634,54,Existing Customer
5,713246208,65,Blue,4548,54,Existing Customer
6,780055308,43,Blue,4482,54,Existing Customer
7,720865383,53,Blue,4437,50,Existing Customer
8,717953283,41,Blue,4379,45,Existing Customer
9,719264058,51,Blue,4305,51,Existing Customer
